In [1]:
import numpy as np 
import pandas as pd 
import re
from nltk.corpus import stopwords
import nltk
from nltk.stem import PorterStemmer
from sklearn.pipeline import Pipeline
from sklearn.base import BaseEstimator, TransformerMixin
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.feature_extraction.text import CountVectorizer
from sklearn.metrics.pairwise import cosine_similarity

nltk.download('stopwords')

[nltk_data] Downloading package stopwords to
[nltk_data]     /home/ammar-y530/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


True

In [2]:
data_path = r"/home/ammar-y530/Documents/S2/SEM 2/RFPP/RFPP_Task/Task N-grams/abstract.csv"
df = pd.read_csv(data_path, sep=";")
df = df.drop(columns=['abstrak_idn'])
df.head()

,title,abstrak_eng
0,Transfer Learning of Pre-trained Transformers ...,"Nowadays, internet has become the most popular..."
1,Personality Classification of Myers Briggs Typ...,Personality classification using textual data ...
2,Developments and Trends in Indonesian Tourism ...,"Information technology has changed society, se..."
3,Offensive Language and Hate Speech Detection U...,Hate speech detection is a crucial issue...
4,IndoBERT Optimization for Sentiment Analysis o...,"In the digital era, sentiment analysis ha..."


In [ ]:
class TextPreprocessor(BaseEstimator, TransformerMixin):
    def __init__(self, stop_words=None):
        self.stop_words = stop_words if stop_words else set()
        self.stemmer = PorterStemmer()

    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        cleaned_texts = []
        for text in X:
            text = text.lower()
            text = re.sub(r'\d+', '', text)
            text = re.sub(r'[^\w\s]', '', text)
            text = re.sub(r'\s+', ' ', text).strip()
            tokens = text.split()
            # tokens = [word for word in tokens if word not in self.stop_words]
            tokens = [self.stemmer.stem(word) for word in tokens]
            # cleaned_texts.append(tokens)
            cleaned_texts.append(" ".join(tokens))
        return cleaned_texts

class TrigramExtractor(BaseEstimator, TransformerMixin):
    def fit(self, X, y=None):
        return self
    
    def transform(self, X):
        trigrams_docs = []
        for tokens in X:
            trigrams = [
                " ".join(tokens[i:i+3])
                for i in range(len(tokens)-2)
            ]
            trigrams_docs.append(trigrams)
        return trigrams_docs

stop_words = set(stopwords.words('english'))

In [4]:
pipeline = Pipeline([
    ("cleaning", TextPreprocessor(stop_words=stop_words)),
    # ("trigram", TrigramExtractor())
])

In [5]:
preprocessed_output = pipeline.fit_transform(df['abstrak_eng'])
print(preprocessed_output)

['nowadays internet has become the most popular source of news however the validity of the online news articles is difficult to assess whether it is a fact or a hoax hoaxes relatedto covidbrought a problematic effect to human life an accurate hoax detection system is important to filter abundant information on the internet in this researcha covid hoax detectionsystemwasproposed by transfer learning of pretrainedtransformer models finetuned original pretrained bert multilingual pretrained mbert and monolingual pretrained indobertwere usedto solve the classification taskin the hoax detection system based on the experimental results finetuned indobert models trained on monolingual indonesian corpus outperform finetuned original and multilingual bert with uncased versions however the finetuned mbert cased model trainedona larger corpus achieved the best performance', 'personality classification using textual data from social media or online forums is a complex task due to the unstructured 

In [6]:
# vectorizer = TfidfVectorizer(ngram_range=(3, 3)) # Pembuatan 3-gram
# tfidf_matrix = vectorizer.fit_transform(preprocessed_output)
# print(tfidf_matrix.shape)

vectorizer = CountVectorizer(ngram_range=(3, 3))  # 3-gram tanpa TF-IDF
count_matrix = vectorizer.fit_transform(preprocessed_output)
print(count_matrix.shape)

(5, 802)


In [12]:
query = "crucial  issue  in  sentiment  analysis  and"
# query = "BERT, hoax, detection"

# stemmed_q = [PorterStemmer().stem(word) for word in query.split()]
# stemmed_q = " ".join(stemmed_q)
# query_vector = vectorizer.transform([stemmed_q])
# scores = cosine_similarity(query_vector, tfidf_matrix).flatten()

stemmed_q = [PorterStemmer().stem(word) for word in query.split()]
stemmed_q = " ".join(stemmed_q)
query_vector = vectorizer.transform([stemmed_q])
scores = cosine_similarity(query_vector, count_matrix).flatten()

In [13]:
print(scores)

[0. 0. 0. 0. 0.]
